<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **Accessing TEMPO NRT (Level 2)**



 <span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> EDL authentication (username/password)

 <span style='color:#ff6666'><font size="5"> **Objectives**

### Subset a remote file

- <font size="3">**a)** By Variables
- <font size="3">**b)** By Spatial selection

### Subset multiple remote files

- <font size="3">Stream subset of data



In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import pydap
import matplotlib.pyplot as plt
# import pydap-specific tools
from pydap.net import create_session
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf
import numpy as np

# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query opendap urls using NASA's CMR API**
    
    
We are interested in `TEMPO NO2 tropospheric and stratospheric columns V04` data. This collection provides hourly data for level 2 data, considered Near Real Time (NRT).


In [ ]:
TEMPO_L2_NRTNO2_ccid = "C3685896872-LARC_CLOUD" # 
time_range = [dt.datetime(2025, 10, 1), dt.datetime(2025, 10, 7)] # One month of data

bounding_box = [-124.63309,46.35932,  -121, 49.83307] # WSEN area within Seattle PNW

cmr_urls = get_cmr_urls(ccid=TEMPO_L2_NRTNO2_ccid, bounding_box=bounding_box, time_range=time_range, limit=1000) # you can incread the limit of results

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

In [ ]:
cmr_urls[:5]

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via earthaccess and OPeNDAP**

<font size="3.5"> You can authenticate via:

* <font size="3.5"> `.netrc` file (username password)
* <font size="3.5"> Token bearer header


<font size="3.5"> OPeNDAP's Hyrax server support both forms of authentication. Below we demonstrate using earthaccess to store and inherit EDL credentials into a session that will be used to stream data from OPeNDAP in the Cloud.


In [ ]:
from earthaccess.exceptions import LoginStrategyUnavailable
try:
    auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials
except LoginStrategyUnavailable:
    auth = earthaccess.login(strategy="interactive", persist=True)

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()


## Inspect Metadata

<font size="3.5"> We can access OPeNDAP produced metadata to identify the variables of interest. In particular those associated with latitude and longitude values


In [ ]:
%%time
pyds = open_url(cmr_urls[0],protocol="dap4", session=my_session)
pyds.tree()

In [ ]:
dims = list(set(pyds['geolocation/latitude'].dims + pyds['geolocation/longitude'].dims + pyds['geolocation/time'].dims))

In [ ]:
print("\nnecessary dimensions to download:", dims, "\n")

### Subset by Variables

<font size="3.5"> We will retain `xtrack`, and all data within Groups `product` and `geolocation`


In [ ]:
output_path = "data/"

## Stream data

<font size="3.5"> Each remote file is stored into an individual file. No data aggregation


In [ ]:
%%time
dap_to_netcdf(cmr_urls, session=my_session, 
              keep_variables= dims + [
                                      "/geolocation/time",
                                      "/geolocation/longitude",
                                      "/geolocation/latitude", 
              ],
              output_path=output_path)

### Inspect local data

In [ ]:
%%time
# Use coord data from Bounding Box
minLon, maxLon = bounding_box[0], bounding_box[2]
minLat, maxLat = bounding_box[1], bounding_box[3]

slices = []
# iterate over all downloaded files
# Will use the URL to extract the filename
for url in cmr_urls[:20]:
    filename = output_path+f"{url.split('/')[-1][:-3]}.nc4"
    # Flatten data 
    ds = xr.merge([xr.open_dataset(filename), xr.open_dataset(filename, group='geolocation')])
    ds.load()
    # Identify subset from Lon/Lat data per granule
    
    longitude = ds['longitude'].values
    latitude = ds['latitude'].values

    mask = (
        (longitude >= minLon) & (longitude <= maxLon) &
        (latitude >= minLat) & (latitude <= maxLat)
    )

    rows, cols = np.where(mask)
    # indexes below
    y0, y1 = rows.min(), rows.max()
    x0, x1 = cols.min(), cols.max()
    slice_ = {
        "mirror_step":(y0,y1),
        "xtrack": (x0,x1),
        }
    slices.append({
        "mirror_step":(y0,y1),
        "xtrack": (x0,x1),
        })

### Visualize Coordinates

<font size="3.5"> Will need to mask arrays for visualizing, 

<font size="3.5"> Plot only the last granule

In [ ]:
Lon = ds['longitude'].isel(mirror_step=slice(y0, y1), xtrack=slice(x0, x1))

Lon_masked = xr.full_like(ds['longitude'], np.nan)
Lon_masked.loc[dict(
    mirror_step=Lon['mirror_step'],
    xtrack=Lon['xtrack']
)] = Lon


Lat = ds['latitude'].isel(mirror_step=slice(y0, y1), xtrack=slice(x0, x1))
Lat_masked = xr.full_like(ds['latitude'], np.nan)
Lat_masked.loc[dict(
    mirror_step=Lat['mirror_step'],
    xtrack=Lat['xtrack']
)] = Lat


In [ ]:
fig, axes = plt.subplots(figsize=(20,8), ncols=2)
pbar_lon = ds['longitude'].plot(ax=axes[0], cmap="Blues", vmin=-160, vmax=-105, levels=np.arange(-160,-105,3), cbar_kwargs={"location": "top"})
pbar_lon.colorbar.ax.tick_params(labelsize=14)
pbar_lon.colorbar.set_label(r'Longitude ($^\circ$E)', fontsize=16, weight='bold')
Lon_masked.plot(ax=axes[0], cmap="Greys_r",vmin=-160,vmax=20, add_colorbar=False, alpha=0.8)

# Optional: Set limits if not automatically handling it
axes[0].set_xlim([ds['xtrack'].min(),ds['xtrack'].max()])
axes[0].set_ylim([ds['mirror_step'].min(),ds['mirror_step'].max()])

pbar_lat = ds['latitude'].plot(ax=axes[1], vmin=15, vmax=62.5, levels=20, cmap='Reds',cbar_kwargs={"location": "top"})
pbar_lat.colorbar.ax.tick_params(labelsize=14)
pbar_lat.colorbar.set_label(r'Latitude ($^\circ$N)', fontsize=16, weight='bold')
Lat_masked.plot(ax=axes[1], cmap="Greys_r",vmin=40,vmax=90, add_colorbar=False, alpha=0.8)


plt.setp(axes[0].get_xticklabels(), fontsize=15)
plt.setp(axes[0].get_yticklabels(), fontsize=15)
axes[0].set_xlabel('xtrack', fontsize=17.5)
axes[0].set_ylabel('mirror_step', fontsize=17.5)

plt.setp(axes[1].get_xticklabels(), fontsize=15)
plt.setp(axes[1].get_yticklabels(), fontsize=15);
axes[1].set_xlabel('xtrack', fontsize=17.5)
axes[1].set_ylabel('mirror_step', fontsize=17.5)
plt.show()

## Now define all variables to download

In [ ]:
Vars = dims + [
    "/product/main_data_quality_flag",
    "/product/vertical_column_troposphere",
    "/product/vertical_column_stratosphere",
    "/geolocation/time",
    "/geolocation/longitude",
    "/geolocation/latitude",
    "/support_data/wind_speed",
    "/support_data/terrain_height",
    "/support_data/gas_profile",
    "/support_data/pbl_height",
    "/support_data/temperature_profile",
]

## Download data

<font size="3.5"> At this moment, need to erase any previously downloaded `TEMPO_NO2_L2_*` data 

<font size="3.5"> to avoid filename collision


In [ ]:
%%time
dap_to_netcdf(cmr_urls, session=my_session, 
              keep_variables = Vars,
              dim_slices= slices,
              output_path=output_path)